# CD8 T Cell Factor Interpretation: Gaublomme Framework

Subset CD8 T cells from all samples, retrain CytoVI with **10 latent factors** on this subset, then systematically interpret each factor by correlating it with:
- **A.** Categorical metadata (condition, time, cell system, sample)
- **B.** Curated marker programs (T cell activation, exhaustion, costimulation, cytotoxicity, memory/differentiation, adhesion/homing)
- **C.** Pseudotime computed via diffusion pseudotime

Each factor is classified as **biology-aligned**, **batch effect**, **novel axis**, or **noise** based on the pattern of associations (Gaublomme et al., Cell 2015).

In [ ]:
# [1] Imports & configuration
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
from scvi.external import CYTOVI
from scvi.external.cytovi import scale as cytovi_scale, transform_arcsinh, merge_batches
import seaborn as sns
import matplotlib.pyplot as plt
import torch

# Local imports — absolute paths for kernel compatibility
_PROJECT_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
_FACTOR_DIR  = _PROJECT_DIR / "factor_analysis"
sys.path.insert(0, str(_FACTOR_DIR))
sys.path.insert(0, str(_PROJECT_DIR))

from factor_utils import (
    MARKER_PROGRAMS, CACHE_DIR, ADATA_PATH, CD8_MODEL_DIR, CD8_ADATA_PATH,
    score_programs, correlate_factors_with_programs, compute_marker_factor_loadings,
    test_factor_metadata_categorical,
    build_factor_metadata_heatmap, build_factor_program_heatmap,
    plot_loadings_heatmap, plot_factor_metadata_violins,
    plot_factor_pseudotime_grid, plot_pseudotime_distributions,
    plot_program_correlation_matrix,
    plot_metadata_scatter_grid, plot_program_scatter_grid, plot_umap_factor_scores,
    select_root_cell, correlate_factors_with_pseudotime,
    build_interpretation_table, plot_factor_scatter,
    display_name, sig_label,
)

sc.set_figure_params(dpi=100, frameon=False)
sns.set_style("whitegrid")

RESULTS_DIR = _FACTOR_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

ASINH_SCALE = 5.0
N_LATENT = 10

print(f"scvi-tools: {scvi.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")


In [ ]:
# [2] Smart load: use cached CD8 adata if available, otherwise run training pipeline
force_retrain = False  # set True to retrain CytoVI from scratch

_use_cache = CD8_ADATA_PATH.exists() and not force_retrain

if _use_cache:
    print(f"Loading cached CD8 adata from {CD8_ADATA_PATH}")
    adata = sc.read_h5ad(CD8_ADATA_PATH)
    factor_scores = adata.obsm["X_CytoVI"]
    print(f"  {adata.n_obs:,} cells | {adata.n_vars} markers | "
          f"{factor_scores.shape[1]} latent factors")
    print("Cells [3]–[5] will be skipped.")
else:
    print("Cache not found or force_retrain=True — will run full training pipeline.")
    # Load full annotated dataset and subset to CD8 T cells
    adata_full = sc.read_h5ad(ADATA_PATH)
    adata_cd8  = adata_full[adata_full.obs["cell_type_annot"] == "CD8"].copy()
    del adata_full
    print(f"CD8 T cells: {adata_cd8.n_obs:,} | Markers: {adata_cd8.n_vars}")
    print(adata_cd8.obs[["condition", "time", "cell_system"]].value_counts().to_string())


In [ ]:
# [3] Per-batch arcsinh transform + min-max scaling  [skipped if using cache]
if not _use_cache:
    batches = adata_cd8.obs["cell_system"].unique().tolist()
    adata_batches = []
    for batch in batches:
        ad = adata_cd8[adata_cd8.obs["cell_system"] == batch].copy()
        ad.layers["raw"] = np.expm1(np.array(ad.layers["log1p"], dtype=np.float32))
        transform_arcsinh(ad, global_scaling_factor=ASINH_SCALE)
        cytovi_scale(ad)
        print(f"  {batch}: {ad.n_obs} cells")
        adata_batches.append(ad)

    adata = merge_batches(adata_batches)
    print(f"Merged: {adata.shape}")

    shared_idx = adata.obs_names.intersection(adata_cd8.obs_names)
    for col in adata_cd8.obs.columns:
        if col not in adata.obs.columns:
            adata.obs[col] = adata_cd8.obs.loc[shared_idx, col]
    for key in adata_cd8.obsm.keys():
        if key not in adata.obsm:
            adata.obsm[key] = adata_cd8[shared_idx].obsm[key]
    for lkey in adata_cd8.layers.keys():
        if lkey not in adata.layers:
            adata.layers[lkey] = adata_cd8[shared_idx].layers[lkey]
    del adata_batches
    print(f"Layers: {list(adata.layers.keys())}")


In [ ]:
# [4] Train CytoVI (n_latent=10)  [skipped if using cache]
if not _use_cache:
    use_gpu = torch.cuda.is_available()
    if CD8_MODEL_DIR.exists() and not force_retrain:
        print(f"Loading cached model from {CD8_MODEL_DIR}")
        CYTOVI.setup_anndata(adata, layer="scaled", batch_key="batch")
        model = CYTOVI.load(CD8_MODEL_DIR, adata=adata)
    else:
        print(f"Training CytoVI (n_latent={N_LATENT}) on {adata.n_obs} CD8 cells...")
        CYTOVI.setup_anndata(adata, layer="scaled", batch_key="batch")
        model = CYTOVI(adata, n_latent=N_LATENT, protein_likelihood="normal")
        model.train(
            batch_size=1024,
            n_epochs_kl_warmup=50,
            accelerator="gpu" if use_gpu else "cpu",
        )
        model.save(CD8_MODEL_DIR, overwrite=True)
        print(f"Model saved → {CD8_MODEL_DIR}")
    print(model)


In [ ]:
# [5] Extract latent representation, build UMAP, cache adata  [skipped if using cache]
if not _use_cache:
    adata.obsm["X_CytoVI"] = model.get_latent_representation()
    adata.layers["imputed"] = model.get_normalized_expression()
    factor_scores = adata.obsm["X_CytoVI"]
    print(f"Latent shape: {factor_scores.shape}")

    sc.pp.neighbors(adata, use_rep="X_CytoVI", n_neighbors=20)
    sc.tl.umap(adata, min_dist=0.3)
    adata.obsm["X_umap_cytovi"] = adata.obsm["X_umap"].copy()

    adata.write_h5ad(CD8_ADATA_PATH)
    del adata_cd8
    print(f"Cached → {CD8_ADATA_PATH}")

# Quick UMAP overview (always runs)
import matplotlib.pyplot as plt
import scanpy as sc
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["condition", "time", "cell_system"]):
    sc.pl.umap(adata, color=col, ax=ax, show=False, title=f"CD8 CytoVI — {col}")
plt.tight_layout()
plt.show()


## Part A: Factor–Metadata Correlations

Test each of the 10 CytoVI factors for association with categorical metadata using Mann-Whitney U (2 groups) or Kruskal-Wallis (>2 groups), with BH-FDR correction.

In [ ]:
# [6] Test factor–metadata associations
cat_cols = ["condition", "time", "cell_system"]
meta_results = test_factor_metadata_categorical(factor_scores, adata.obs, cat_cols)
print(f"Total tests: {len(meta_results)}")
display(meta_results.sort_values("padj").head(20))

In [ ]:
# [7] Heatmap: factor–metadata effect sizes
fig, ax = build_factor_metadata_heatmap(meta_results, metric="effect_size")
fig.savefig(RESULTS_DIR / "factor_metadata_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# [8] Violin plots for top factor–metadata associations
fig, axes = plot_factor_metadata_violins(factor_scores, meta_results, adata.obs, top_n=6)
fig.savefig(RESULTS_DIR / "factor_metadata_violins.pdf", bbox_inches="tight")
plt.show()


## Part B: Factor–Marker Program Correlations

Score each CD8 T cell on 6 curated marker programs (activation, exhaustion, costimulation, cytotoxicity, memory/differentiation, adhesion/homing), then correlate program scores with each CytoVI factor. Also compute proxy loadings (marker–factor Pearson r) since CytoVI has no direct gene loadings like PCA.

In [ ]:
# [9] Marker programs used in this analysis
for name, markers in MARKER_PROGRAMS.items():
    present = [m for m in markers if m in adata.var_names]
    aliases = [display_name(m) for m in present]
    print(f"{name} ({len(present)}/{len(markers)}): {', '.join(aliases)}")

In [ ]:
# [10] Score programs and correlate with factors
program_scores = score_programs(adata, MARKER_PROGRAMS, layer="arcsinh")
r_vals, p_vals = correlate_factors_with_programs(factor_scores, program_scores)

print("Program score summary:")
display(program_scores.describe().round(3))

In [ ]:
# [11] Heatmap: factor–program correlations
fig, ax = build_factor_program_heatmap(r_vals, p_vals)
fig.savefig(RESULTS_DIR / "factor_program_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# [12] Proxy loadings: marker–factor Pearson r
loadings = compute_marker_factor_loadings(adata, factor_scores, layer="arcsinh")

# Show top 5 markers per factor (by absolute correlation)
print("Top 5 markers per factor (proxy loadings):\n")
for col in loadings.columns:
    top = loadings[col].abs().nlargest(5)
    markers_str = ", ".join(
        f"{display_name(m)} ({loadings.loc[m, col]:+.3f})" for m in top.index
    )
    print(f"  {col}: {markers_str}")

In [ ]:
# [13] Loadings heatmap grouped by marker category
fig, g = plot_loadings_heatmap(loadings, programs=MARKER_PROGRAMS)
fig.savefig(RESULTS_DIR / "proxy_loadings_heatmap.pdf", bbox_inches="tight")
plt.show()


## Part C: Pseudotime Analysis

Compute diffusion pseudotime directly on CD8 T cells using CytoVI latent space for neighbors. Root cell is selected as the most naive CD8 T cell (highest CD45RA, lowest activation markers). Correlate pseudotime with each factor to identify differentiation-associated dimensions.

In [ ]:
# [14] Diffusion pseudotime on CD8 cells
# Neighbors are already computed from CytoVI latent above
sc.tl.diffmap(adata)

# Select root cell (most naive CD8 T cell)
root_name = select_root_cell(adata, layer="arcsinh")
adata.uns["iroot"] = np.where(adata.obs_names == root_name)[0][0]
print(f"Root cell: {root_name}")

# Compute diffusion pseudotime
sc.tl.dpt(adata)
print(f"DPT range: {adata.obs['dpt_pseudotime'].min():.3f} – {adata.obs['dpt_pseudotime'].max():.3f}")


In [ ]:
# [15] Correlate pseudotime with factors
pt_corr = correlate_factors_with_pseudotime(factor_scores, adata.obs["dpt_pseudotime"])
print("Factor–Pseudotime correlations (sorted by |Spearman r|):")
display(pt_corr.sort_values("spearman_r", key=abs, ascending=False))

In [ ]:
# [16] Factor scores along pseudotime (2×5 grid)
fig, axes = plot_factor_pseudotime_grid(
    factor_scores, adata.obs["dpt_pseudotime"].values, pt_corr,
    condition=adata.obs["condition"].values, n_latent=N_LATENT,
)
fig.savefig(RESULTS_DIR / "factor_pseudotime_grid.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# [17] Pseudotime distributions by condition and time
fig, axes = plot_pseudotime_distributions(
    adata.obs["dpt_pseudotime"].values, adata.obs
)
fig.savefig(RESULTS_DIR / "pseudotime_by_condition.pdf", bbox_inches="tight")
plt.show()


## Part D: Interpretation Summary

Classify each factor using the Gaublomme decision matrix:
| Metadata sig? | Program sig? | Classification |
|---|---|---|
| Yes | Yes | Biology aligned with design |
| Yes | No | Possible batch effect |
| No | Yes | Novel axis (undiscovered biology) |
| No | No | Noise |

In [ ]:
# [18] Interpretation summary table
summary = build_interpretation_table(r_vals, p_vals, meta_results)
display(summary)

# Count classifications
print("\nClassification counts:")
print(summary["classification"].value_counts().to_string())

In [ ]:
# [19] Program–program correlation matrix (orthogonality check)
fig, ax = plot_program_correlation_matrix(program_scores)
fig.savefig(RESULTS_DIR / "program_correlation_matrix.pdf", bbox_inches="tight")
plt.show()


## Part E: Visualization

Factor-space scatter plots and UMAP colored by factor scores.

In [ ]:
# [20] Factor-space scatter coloured by metadata
total_abs_r = r_vals.abs().sum(axis=1)
top2 = total_abs_r.nlargest(2).index.tolist()
fx, fy = int(top2[0][1:]), int(top2[1][1:])
print(f"Most informative factors: F{fx} (sum|r|={total_abs_r[top2[0]]:.2f}), "
      f"F{fy} (sum|r|={total_abs_r[top2[1]]:.2f})")

fig, axes = plot_metadata_scatter_grid(
    factor_scores, fx, fy, adata.obs,
    meta_cols=["condition", "time", "cell_system"],
)
fig.savefig(RESULTS_DIR / "factor_scatter_metadata.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# [21] Factor-space scatter coloured by program scores
fig, axes = plot_program_scatter_grid(factor_scores, fx, fy, program_scores)
fig.savefig(RESULTS_DIR / "factor_scatter_programs.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# [22] UMAP coloured by each factor score (2×5 grid)
umap_key = "X_umap_cytovi" if "X_umap_cytovi" in adata.obsm else "X_umap"
fig, axes = plot_umap_factor_scores(
    adata.obsm[umap_key], factor_scores, n_latent=N_LATENT, umap_key=umap_key
)
fig.savefig(RESULTS_DIR / "umap_factor_scores.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# [23] Save tabular results
summary.to_csv(RESULTS_DIR / "factor_interpretation_summary.csv", index=False)
r_vals.to_csv(RESULTS_DIR / "factor_program_correlations.csv")
p_vals.to_csv(RESULTS_DIR / "factor_program_pvalues.csv")
meta_results.to_csv(RESULTS_DIR / "factor_metadata_tests.csv", index=False)
loadings.to_csv(RESULTS_DIR / "proxy_loadings.csv")
pt_corr.to_csv(RESULTS_DIR / "factor_pseudotime_correlations.csv", index=False)

print("Results saved to:", RESULTS_DIR.resolve())
print("Files:", [f.name for f in RESULTS_DIR.glob("*.csv")])